# Reproduce the benchmark (Table 2) on Colab

4 systems x 2 trained arms x 5 seeds = **40 runs**, using the **release scripts themselves**
rather than the common harness of the sweep notebooks. Those sweeps trade absolute accuracy
for a controlled comparison; this notebook is the opposite trade.

| | expected (Table 2) | | |
|---|--:|--:|--:|
| | **Arm 1** | **Arm 2** | **Arm 3** |
| SDOF oscillator | +0.7234 ± 0.0682 | +0.3014 ± 0.0460 | +0.9123 ± 0.0259 |
| 3-storey steel frame | +0.7516 ± 0.0441 | +0.7065 ± 0.0351 | +0.8363 ± 0.0351 |
| Wind turbine | +0.2730 ± 0.0312 | +0.3061 ± 0.1310 | +0.8631 ± 0.0168 |
| Buried tunnel | −0.0689 ± 0.0594 | +0.7292 ± 0.0768 | +0.8704 ± 0.0648 |

R² is pooled for the first two rows and the unweighted mean over sensors for the last two.

**Budget several hours and expect not to finish in one session.** Every run writes its own
JSON; re-running the sweep cell skips what is on disk.


## 1 · GPU check

In [ ]:
import torch, platform
assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all.")
print(torch.cuda.get_device_name(0))
print("torch", torch.__version__, "| python", platform.python_version())

## 2 · Point at the data

In [ ]:
import os, shutil

from google.colab import drive
drive.mount("/content/drive")

# ── edit this one line if you put the folder somewhere else ──
BASE = "/content/drive/MyDrive/colab_benchmark"

OUTD = os.path.join(BASE, "results")
os.makedirs(OUTD, exist_ok=True)

# The release scripts read their databases from the working directory by name, so stage
# everything into one local folder and run there.
WORK = "/content/bench"
os.makedirs(WORK, exist_ok=True)
for sub in ("data", "scripts"):
    src = os.path.join(BASE, sub)
    assert os.path.isdir(src), f"missing {src}"
    for f in os.listdir(src):
        d = os.path.join(WORK, f)
        if not os.path.exists(d) or os.path.getsize(d) != os.path.getsize(os.path.join(src, f)):
            shutil.copy2(os.path.join(src, f), d)
os.chdir(WORK)
print("staged:", sorted(os.listdir(WORK)))
print("results ->", OUTD)

## 3 · Configuration

`SPEC` holds, per (system, arm), the script to run, the environment it needs, and how to
read its metric back — the four scripts report in four different ways.


In [ ]:
SEEDS = [42, 0, 1, 2, 3]

SPEC = {
    ("sdof", 3):    dict(cmd=["python3", "sdof_arm3.py"],
                         env={"DB": "sdof_eta60_n1500.npz"}, read="sdof3"),
    ("sdof", 2):    dict(cmd=["python3", "sdof_arm2.py", "SDOF"],
                         env={"DB": "sdof_eta60_n1500.npz", "ARM2_FEEDBACK": "0"},
                         read="sdof2"),
    ("frame3", 3):  dict(cmd=["python3", "frame_run.py", "{seed}", "3", "10"],
                         env={}, read="frame"),
    ("frame3", 2):  dict(cmd=["python3", "frame_run.py", "{seed}", "2", "10"],
                         env={}, read="frame"),
    ("tunnel", 3):  dict(cmd=["python3", "tunnel_arm3.py"],
                         env={"N_MAX": "500", "TAG": "bench_tunnel_arm3_s{seed}"},
                         read="csv", csv="bench_tunnel_arm3_s{seed}_results.csv",
                         metric="unweighted"),
    ("tunnel", 2):  dict(cmd=["python3", "tunnel_arm2.py"],
                         env={"N_MAX": "500", "TAG": "bench_tunnel_arm2_s{seed}"},
                         read="csv", csv="bench_tunnel_arm2_s{seed}_results.csv",
                         metric="unweighted"),
    ("turbine", 3): dict(cmd=["python3", "turbine_arm3.py"],
                         env={"DB_PATH": "wt_v20_full.npz", "N_MAX": "0", "FLIP": "1",
                              "STEP": "50", "SAMPLE_W": "0",
                              "TAG": "bench_wt_arm3_s{seed}"},
                         read="csv", csv="bench_wt_arm3_s{seed}_results.csv",
                         metric="unweighted"),
    ("turbine", 2): dict(cmd=["python3", "turbine_arm2.py"],
                         env={"DB_PATH": "wt_v20_full.npz", "N_MAX": "0", "FLIP": "1",
                              "STEP": "50", "TAG": "bench_wt_arm2_s{seed}"},
                         read="csv", csv="bench_wt_arm2_s{seed}_results.csv",
                         metric="unweighted"),
}
SYSTEMS = ["sdof", "frame3", "turbine", "tunnel"]
print(f"{len(SYSTEMS)} systems x 2 arms x {len(SEEDS)} seeds = "
      f"{len(SYSTEMS)*2*len(SEEDS)} runs")

## 4 · Run

Each run is a subprocess, so a script that dies cannot take the notebook with it. Its full
log is kept beside the JSON, which is what you will need if a number comes out wrong.


In [ ]:
import csv, json, os, re, subprocess, time
import statistics as st

def key(ex, arm, seed):
    return f"{ex}_arm{arm}_s{seed}"

def finished(ex, arm, seed):
    jf = os.path.join(OUTD, key(ex, arm, seed) + ".json")
    if not os.path.exists(jf):
        return False
    try:
        r = json.load(open(jf))
        return "arm" in r and "arm1" in r
    except Exception:
        print(f"   corrupt {os.path.basename(jf)} -> will re-run"); return False

def save_atomic(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as fh:
        json.dump(obj, fh, indent=1); fh.flush(); os.fsync(fh.fileno())
    os.replace(tmp, path)

def read_result(spec, seed, log):
    """-> (arm1, armN). Four scripts, four reporting conventions."""
    how = spec["read"]
    if how == "sdof3":
        # "  Global R2                      0.7335          0.9250"
        m = re.search(r"Global R.{0,3}\s+([+-]?\d+\.\d+)\s+([+-]?\d+\.\d+)", log)
        assert m, "SDOF arm3: could not find the Global R2 line"
        return float(m.group(1)), float(m.group(2))
    if how == "sdof2":
        # "  SDOF u: R2=0.3736 peak%=... " ; Arm 1 comes from the arm3 run of the same seed
        m = re.search(r"SDOF\s+u:\s*R2=([+-]?\d+\.\d+)", log)
        assert m, "SDOF arm2: could not find the R2 line"
        return None, float(m.group(1))
    if how == "frame":
        m = re.search(r"SUMMARY seed=\S+ arm1=([+-]\d+\.\d+) arm\d=([+-]\d+\.\d+)", log)
        assert m, "frame: could not find the SUMMARY line"
        return float(m.group(1)), float(m.group(2))
    if how == "csv":
        path = spec["csv"].format(seed=seed)
        rows = {r["label"]: r for r in csv.DictReader(open(path))}
        def val(r):
            if spec.get("metric") == "unweighted":
                return st.mean([float(r[f"r2_s{i}"]) for i in range(1, 12)
                                if f"r2_s{i}" in r])
            return float(r["r2_overall"])
        return val(rows["u_red_ref"]), val(rows["trained"])
    raise ValueError(how)

todo = [(e, a, s) for e in SYSTEMS for a in (3, 2) for s in SEEDS]
pre = [t for t in todo if finished(*t)]
print(f"{len(pre)}/{len(todo)} already done, {len(todo)-len(pre)} to go\n")
t_start = time.perf_counter(); n_new = 0

for n, (ex, arm, seed) in enumerate(todo, 1):
    tag = f"{ex}/arm{arm}/s{seed}"
    if finished(ex, arm, seed):
        print(f"[{n:2d}/{len(todo)}] {tag:22s} SKIP (done)"); continue
    spec = SPEC[(ex, arm)]
    cmd = [c.format(seed=seed) for c in spec["cmd"]]
    env = {**os.environ, "SEED": str(seed), "MPLBACKEND": "Agg",
           **{k: v.format(seed=seed) for k, v in spec["env"].items()}}
    print(f"[{n:2d}/{len(todo)}] {tag:22s} {' '.join(cmd)} ...", flush=True)
    t0 = time.perf_counter()
    p = subprocess.run(cmd, env=env, capture_output=True, text=True)
    log = p.stdout + "\n" + p.stderr
    open(os.path.join(OUTD, key(ex, arm, seed) + ".log"), "w").write(log)
    if p.returncode != 0:
        print(f"   !! exit {p.returncode}; see {key(ex,arm,seed)}.log\n"); continue
    try:
        a1, aN = read_result(spec, seed, log)
    except Exception as exc:
        print(f"   !! could not parse: {exc}\n"); continue
    save_atomic(dict(example=ex, arm=arm, seed=seed, arm1=a1, armN=aN,
                     sec=round(time.perf_counter() - t0, 1)),
                os.path.join(OUTD, key(ex, arm, seed) + ".json"))
    n_new += 1
    el = time.perf_counter() - t_start
    print(f"   arm1={a1 if a1 is None else round(a1,4)}  arm{arm}={aN:+.4f}  "
          f"{time.perf_counter()-t0:.0f}s | ETA "
          f"{(len(todo)-len(pre)-n_new)*el/n_new/60:.0f}m\n", flush=True)

nd = sum(finished(*t) for t in todo)
print(f"{nd}/{len(todo)} complete. {(time.perf_counter()-t_start)/60:.0f} min this session.")
if nd < len(todo):
    print("Not finished -- re-run cells 1-3 then this cell to continue.")

## 5 · The table

Arm 1 is taken from each system's Arm 3 run, which reports it on the same split. The SDOF
Arm 2 script does not print Arm 1, which is why it is read from the Arm 3 run of the same
seed rather than from its own log.


In [ ]:
import csv, glob, json, os
import statistics as st

R = {}
for f in glob.glob(os.path.join(OUTD, "*_arm*_s*.json")):
    r = json.load(open(f)); R[(r["example"], r["arm"], r["seed"])] = r

NICE = {"sdof": "SDOF oscillator", "frame3": "3-storey steel frame",
        "turbine": "Wind turbine on a monopile", "tunnel": "Buried tunnel"}
REF = {"sdof": (0.7234, 0.3014, 0.9123), "frame3": (0.7516, 0.7065, 0.8363),
       "turbine": (0.2730, 0.3061, 0.8631), "tunnel": (-0.0689, 0.7292, 0.8704)}

out = [["System", "Arm 1", "Arm 2", "Arm 3",
        "Arm 1 (Table 2)", "Arm 2 (Table 2)", "Arm 3 (Table 2)"]]
print(f"{'System':26} {'Arm 1':>17} {'Arm 2':>17} {'Arm 3':>17}")
for ex in SYSTEMS:
    got, miss = {}, False
    for arm in (1, 2, 3):
        if arm == 1:
            v = [R[(ex, 3, s)]["arm1"] for s in SEEDS if (ex, 3, s) in R]
        else:
            v = [R[(ex, arm, s)]["armN"] for s in SEEDS if (ex, arm, s) in R]
        v = [x for x in v if x is not None]
        if len(v) < len(SEEDS):
            miss = True
        got[arm] = v
    if miss:
        print(f"{NICE[ex]:26} incomplete: "
              + " ".join(f"arm{a} {len(got[a])}/{len(SEEDS)}" for a in (1, 2, 3)))
        out.append([NICE[ex]] + ["-"] * 6); continue
    cells = [f"{st.mean(got[a]):+.4f} +/- {st.stdev(got[a]):.4f}" for a in (1, 2, 3)]
    out.append([NICE[ex]] + cells + [f"{x:+.4f}" for x in REF[ex]])
    print(f"{NICE[ex]:26} " + " ".join(f"{c:>17}" for c in cells))
    print(f"{'  vs Table 2':26} "
          + " ".join(f"{st.mean(got[a])-REF[ex][a-1]:>+17.4f}" for a in (1, 2, 3)))

p = os.path.join(OUTD, "table2_reproduction.csv")
with open(p, "w", newline="") as fh:
    csv.writer(fh).writerows(out)
print("\nwrote", p)

---

### What was patched, and why

**SDOF, both arms.** `SEED` was a module-level literal, so the environment variable was
ignored and all five seeds trained the same split. That is a defect in the archived scripts,
not a Colab accommodation — the same bug is patched in the web application's loader.

**Frame runner.** It forced `device = cpu` and rewrote the notebook's CUDA assertion, because
it was written for a local machine. Those three substitutions are removed so it uses the T4.
Everything else — the CPU-affordable override L = 150, STEP = 150, HIDDEN = 128, LAYERS = 1,
BATCH = 64, EPOCHS = 10 — is left exactly as the published run used it, since that override
is part of what produced 0.8363.

Both patch sets are asserted to apply exactly once in `make_colab_benchmark.py`.

### If a number does not match

Each run's full log is saved next to its JSON. The likeliest causes, in order: a different
`N_MAX` on the tunnel (it must be 500, not the full 1000), the turbine reading the wrong
database (it must be `wt_v20_full.npz`, the V_hub-capped set), and non-determinism from GPU
thread counts, which moved the tunnel by about 0.03 when it was reproduced locally.

### Scope

This reproduces Table 2 only. The sweep notebooks deliberately use a uniform reduced harness
so that a difference between two columns is attributable to the swept variable; this one
deliberately does not, so its numbers are comparable to the paper but its four systems are
not comparable to each other.
